In [ ]:

import zipfile
import os
import pandas as pd
from sklearn.model_selection import train_test_split


# Load your CSVs
df_real = pd.read_csv("/kaggle/input/my-dataset/LabeledAuthentic-7K.csv")
df_fake = pd.read_csv("/kaggle/input/my-dataset/LabeledFake-1K.csv")

df_fake_aug=pd.read_csv("/kaggle/input/fixed-dataset/augmented_SBERT_top3(1).csv")



# --- CHANGE IS HERE: KEEP 'articleID' ---
# Rename articleID to article_id for consistency if needed, or just keep it
df_real = df_real[[ 'content', 'label', 'category']]
df_fake = df_fake[[ 'content', 'label', 'category']]

df_fake_aug=df_fake_aug[[ 'content', 'label', 'category']]
# Split each dataset individually: 70% train, 30% test
train_real, test_real = train_test_split(df_real, test_size=0.3, random_state=50, stratify=df_real['label'])
train_fake, test_fake = train_test_split(df_fake, test_size=0.3, random_state=50, stratify=df_fake['label'])

train_fake=pd.concat([train_fake,df_fake_aug]).sample(frac=1,random_state=50)

# Concatenate train splits and test splits
train_df = pd.concat([train_real, train_fake]).sample(frac=1, random_state=50).reset_index(drop=True)
test_df = pd.concat([test_real, test_fake]).sample(frac=1, random_state=50).reset_index(drop=True)



In [ ]:
# Optional: check sizes and distribution
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
print(f"Train label distribution:\n{train_df['label'].value_counts()}")
print(f"Test label distribution:\n{test_df['label'].value_counts()}")

Train size: 8671, Test size: 2551
Train label distribution:
label
1.0    5041
0.0    3630
Name: count, dtype: int64
Test label distribution:
label
1.0    2161
0.0     390
Name: count, dtype: int64


In [ ]:
# Cell 2: Tokenization and Dataset
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

MODEL_NAME = "sagorsarker/bangla-bert-base"
MAX_LENGTH = 512
BATCH_SIZE = 8

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = FakeNewsDataset(train_df['content'].tolist(), train_df['label'].tolist(), tokenizer, MAX_LENGTH)
test_dataset = FakeNewsDataset(test_df['content'].tolist(), test_df['label'].tolist(), tokenizer, MAX_LENGTH)

# Optional: create a dev set (10% of training data)
from sklearn.model_selection import train_test_split
train_texts, dev_texts, train_labels, dev_labels = train_test_split(
    train_df['content'].tolist(), train_df['label'].tolist(), test_size=0.1, random_state=50, stratify=train_df['label']
)
dev_dataset = FakeNewsDataset(dev_texts, dev_labels, tokenizer, MAX_LENGTH)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


In [ ]:
# Cell 3: Model & Training Setup
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW   # <-- change here
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 3


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sagorsarker/bangla-bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from tqdm import tqdm

# Ensure optimizer is defined before this loop (e.g., optimizer = AdamW(model.parameters(), lr=...))

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0

    # Progress bar for the training batches
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # 1. Zero gradients
        optimizer.zero_grad()

        # 2. Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # 3. Backward pass
        loss.backward()

        # 4. Update weights
        optimizer.step()

        train_loss += loss.item()

    # Calculate average loss for the epoch (Indentation matches 'for epoch')
    avg_train_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Train Loss: {avg_train_loss:.4f}")

    # --- VALIDATION PHASE (Dev Set) ---
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print(f"Dev Accuracy: {correct/total:.4f}")

Training Epoch 1: 100%|██████████| 1084/1084 [16:52<00:00,  1.07it/s]


Epoch 1 | Avg Train Loss: 0.1778
Dev Accuracy: 0.9758


Training Epoch 2: 100%|██████████| 1084/1084 [16:51<00:00,  1.07it/s]


Epoch 2 | Avg Train Loss: 0.0648
Dev Accuracy: 0.9827


Training Epoch 3: 100%|██████████| 1084/1084 [16:51<00:00,  1.07it/s]


Epoch 3 | Avg Train Loss: 0.0322
Dev Accuracy: 0.9919


In [ ]:
# Cell 5: Test Evaluation
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds))


              precision    recall  f1-score   support

           0       0.80      0.91      0.85       390
           1       0.98      0.96      0.97      2161

    accuracy                           0.95      2551
   macro avg       0.89      0.93      0.91      2551
weighted avg       0.95      0.95      0.95      2551



In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

# 1. Calculate the Overall Accuracy
overall_accuracy = accuracy_score(all_labels, all_preds)

# 2. Calculate Precision, Recall, and F1 for each class (0 and 1)
precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, labels=[0, 1])

# 3. Calculate Weighted Averages
precision_avg, recall_avg, f1_avg, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

# --- FIX START ---
# We use recall[0] as the "Accuracy" for class 0, and recall[1] for class 1.
# This represents: (Correctly Predicted X) / (Total Actual X)

results_data = {
    'Category': ['Fake News (Label 0)', 'Real News (Label 1)', 'Combined (Real + Fake)'],

    # CHANGE HERE: Replaced '-' with the actual recall variables
    'Accuracy': [f"{recall[0]:.4f}", f"{recall[1]:.4f}", f"{overall_accuracy:.4f}"],

    'Precision': [f"{precision[0]:.4f}", f"{precision[1]:.4f}", f"{precision_avg:.4f}"],
    'Recall': [f"{recall[0]:.4f}", f"{recall[1]:.4f}", f"{recall_avg:.4f}"],
    'F1 Score': [f"{f1[0]:.4f}", f"{f1[1]:.4f}", f"{f1_avg:.4f}"]
}
# --- FIX END ---

df_results = pd.DataFrame(results_data)

# 5. Display the table
print("\nDetailed Performance Metrics:")
display(df_results)

# 6. OPTIONAL: Display Confusion Matrix (Best way to see exactly where mistakes happen)
cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(f"True Negatives (Correct Fake): {cm[0][0]}")
print(f"False Positives (Fake called Real): {cm[0][1]}")
print(f"False Negatives (Real called Fake): {cm[1][0]}")
print(f"True Positives (Correct Real): {cm[1][1]}")


Detailed Performance Metrics:


,Category,Accuracy,Precision,Recall,F1 Score
0,Fake News (Label 0),0.9051,0.7968,0.9051,0.8475
1,Real News (Label 1),0.9584,0.9824,0.9584,0.9703
2,Combined (Real + Fake),0.9502,0.9541,0.9502,0.9515



Confusion Matrix:
True Negatives (Correct Fake): 353
False Positives (Fake called Real): 37
False Negatives (Real called Fake): 90
True Positives (Correct Real): 2071
